# 03: Transformation & Encoding

In this notebook, you'll learn to transform data for machine learning: scaling, encoding, text processing, and feature engineering.

## Learning Objectives

- Apply normalization (MinMaxScaler) and standardization (StandardScaler)
- Encode categorical variables with one-hot encoding and label encoding
- Process text data: lowercasing, tokenization, stopword removal
- Create new features from existing data
- Parse and extract information from date/time columns

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re

print(f"Pandas version: {pd.__version__}")

## 2. Sample Dataset

In [ ]:
# Create a sample employee dataset
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Henry"],
    "age": [28, 35, 42, 31, 26, 45, 33, 29],
    "salary": [75000, 82000, 95000, 68000, 72000, 110000, 88000, 71000],
    "department": ["Engineering", "Marketing", "Engineering", "HR", 
                   "Marketing", "Engineering", "HR", "Marketing"],
    "size": ["S", "M", "L", "M", "S", "XL", "M", "L"],
    "join_date": ["2021-03-15", "2020-07-01", "2019-01-20", "2022-11-05",
                   "2023-02-14", "2018-06-30", "2021-09-12", "2022-04-22"],
    "review": [
        "Great employee, very productive and reliable",
        "Good work but needs to improve communication",
        "Excellent performance, exceeds expectations",
        "Average worker, meets basic requirements",
        "Outgoing personality, great team player",
        "Highly skilled, delivers quality results",
        "Needs improvement in time management",
        "Solid performer, consistent output"
    ]
})

df

## 3. Normalization (Min-Max Scaling)

Scales values to the range [0, 1]. Use when you need values on the same scale.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Select numeric columns to normalize
numeric_cols = ["age", "salary"]

# Apply MinMaxScaler
df_normalized = df.copy()
df_normalized[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("=== Before Normalization ===")
print(df[numeric_cols])

print("\n=== After Normalization ===")
print(df_normalized[numeric_cols].round(3))

print(f"\nMin values: {df_normalized[numeric_cols].min().tolist()}")
print(f"Max values: {df_normalized[numeric_cols].max().tolist()}")

## 4. Standardization (Z-score Scaling)

Centers data around mean=0 with std=1. Good for algorithms that assume normal distribution.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_standardized = df.copy()
df_standardized[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("=== Before Standardization ===")
print(df[numeric_cols])

print("\n=== After Standardization ===")
print(df_standardized[numeric_cols].round(3))

print(f"\nMean: {df_standardized[numeric_cols].mean().tolist()}")
print(f"Std:  {df_standardized[numeric_cols].std().round(3).tolist()}")

**When to use which:**
- **MinMaxScaler**: Neural networks, algorithms sensitive to magnitude, bounded ranges needed
- **StandardScaler**: Linear regression, SVM, PCA, algorithms assuming normal distribution

## 5. One-Hot Encoding

Converts categorical variables into binary (0/1) columns. Use when categories have no order.

In [ ]:
# One-hot encode department
df_encoded = pd.get_dummies(df, columns=["department"], prefix="dept")
print("=== Original ===")
print(df[["name", "department"]])

print("\n=== After One-Hot Encoding ===")
df_encoded

In [ ]:
# Drop first category to avoid multicollinearity (dummy trap)
df_encoded_drop = pd.get_dummies(df, columns=["department"], prefix="dept", drop_first=True)
print("With drop_first=True:")
df_encoded_drop[["name", "dept_Engineering", "dept_HR"]]

## 6. Label Encoding

Assigns an integer to each category. Use for ordinal data where order matters.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encode size (ordinal: S < M < L < XL)
le = LabelEncoder()

df_label = df.copy()
df_label["size_encoded"] = le.fit_transform(df_label["size"])

print("=== Label Encoding for Size ===")
print(df_label[["name", "size", "size_encoded"]])
print(f"\nMapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Manual ordinal encoding (preserve natural order)
size_order = {"S": 0, "M": 1, "L": 2, "XL": 3}
df_label["size_ordinal"] = df_label["size"].map(size_order)

print("=== Manual Ordinal Encoding ===")
print(df_label[["size", "size_encoded", "size_ordinal"]])

**One-Hot vs Label Encoding:**
- **One-Hot**: No natural order (department, color, country). Creates multiple columns.
- **Label/Ordinal**: Natural order exists (S < M < L). Single column.

## 7. Text Processing

In [ ]:
# Sample reviews
reviews = df["review"].tolist()
print("Original reviews:")
for i, r in enumerate(reviews):
    print(f"  {i}: {r}")

### 7.1 Lowercasing

In [ ]:
df_text = df.copy()
df_text["review_lower"] = df_text["review"].str.lower()

print("=== Lowercased ===")
for i in range(3):
    print(f"  {df_text['review'].iloc[i]}")
    print(f"  -> {df_text['review_lower'].iloc[i]}")
    print()

### 7.2 Tokenization

In [ ]:
# Tokenize: split text into words
df_text["tokens"] = df_text["review_lower"].str.split()

print("=== Tokenized ===")
for i in range(3):
    print(f"  {df_text['tokens'].iloc[i]}")

### 7.3 Remove Punctuation and Special Characters

In [ ]:
# Remove punctuation from tokens
df_text["tokens_clean"] = df_text["tokens"].apply(
    lambda tokens: [re.sub(r"[^\w]", "", t) for t in tokens]
)

# Remove empty strings that result from removing punctuation
df_text["tokens_clean"] = df_text["tokens_clean"].apply(
    lambda tokens: [t for t in tokens if t]
)

print("=== Cleaned Tokens ===")
for i in range(3):
    print(f"  {df_text['tokens_clean'].iloc[i]}")

### 7.4 Stopword Removal

In [ ]:
# Common English stopwords
stopwords = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing",
    "a", "an", "the", "and", "but", "if", "or", "because", "as",
    "until", "while", "of", "at", "by", "for", "with", "about",
    "to", "from", "in", "out", "on", "off", "over", "under",
    "very", "can", "will", "just", "should", "now"
}

df_text["tokens_nostop"] = df_text["tokens_clean"].apply(
    lambda tokens: [t for t in tokens if t not in stopwords]
)

print("=== Without Stopwords ===")
for i in range(3):
    print(f"  Before: {df_text['tokens_clean'].iloc[i]}")
    print(f"  After:  {df_text['tokens_nostop'].iloc[i]}")
    print()

### 7.5 Reconstruct Text

In [ ]:
# Join tokens back into text
df_text["review_processed"] = df_text["tokens_nostop"].str.join(" ")

print("=== Processed Reviews ===")
for i in range(len(df_text)):
    print(f"  Original:  {df_text['review'].iloc[i]}")
    print(f"  Processed: {df_text['review_processed'].iloc[i]}")
    print()

## 8. Feature Engineering

### 8.1 From Numeric Data

In [ ]:
df_feat = df.copy()

# Age bins
df_feat["age_group"] = pd.cut(
    df_feat["age"], 
    bins=[0, 18, 35, 60, 100],
    labels=["child", "young_adult", "adult", "senior"]
)

# Salary bins
df_feat["salary_level"] = pd.cut(
    df_feat["salary"],
    bins=[0, 50000, 80000, 100000, 200000],
    labels=["low", "medium", "high", "very_high"]
)

print("=== Binning ===")
df_feat[["name", "age", "age_group", "salary", "salary_level"]]

In [ ]:
# Log transform for skewed data
df_feat["log_salary"] = np.log1p(df_feat["salary"])

print("=== Log Transform ===")
print(df_feat[["name", "salary", "log_salary"]])

### 8.2 From Text Data

In [ ]:
df_feat["review_length"] = df_feat["review"].str.len()
df_feat["word_count"] = df_feat["review"].str.split().str.len()
df_feat["has_positive"] = df_feat["review"].str.contains(
    "great|excellent|outstanding|good|solid", case=False
).astype(int)
df_feat["has_negative"] = df_feat["review"].str.contains(
    "needs|improve|poor|bad", case=False
).astype(int)

print("=== Text Features ===")
df_feat[["name", "review_length", "word_count", "has_positive", "has_negative"]]

### 8.3 From Date/Time Data

In [ ]:
df_feat["join_date"] = pd.to_datetime(df_feat["join_date"])

df_feat["join_year"] = df_feat["join_date"].dt.year
df_feat["join_month"] = df_feat["join_date"].dt.month
df_feat["join_day_of_week"] = df_feat["join_date"].dt.dayofweek
df_feat["is_weekend_join"] = df_feat["join_day_of_week"].isin([5, 6]).astype(int)

# Tenure in days (from join date to today)
df_feat["tenure_days"] = (pd.Timestamp.now() - df_feat["join_date"]).dt.days

print("=== Date Features ===")
df_feat[["name", "join_date", "join_year", "join_month", 
         "join_day_of_week", "is_weekend_join", "tenure_days"]]

## 9. Complete Transformation Pipeline

In [ ]:
def transform_for_ml(df):
    """Transform raw data into ML-ready features."""
    df = df.copy()
    
    # --- Numeric scaling ---
    scaler = StandardScaler()
    df[["age_scaled", "salary_scaled"]] = scaler.fit_transform(df[["age", "salary"]])
    
    # --- Categorical encoding ---
    df = pd.get_dummies(df, columns=["department"], prefix="dept", drop_first=True)
    
    # --- Ordinal encoding ---
    size_map = {"S": 0, "M": 1, "L": 2, "XL": 3}
    df["size_encoded"] = df["size"].map(size_map)
    
    # --- Date features ---
    df["join_date"] = pd.to_datetime(df["join_date"])
    df["join_year"] = df["join_date"].dt.year
    df["join_month"] = df["join_date"].dt.month
    df["is_weekend_join"] = df["join_date"].dt.dayofweek.isin([5, 6]).astype(int)
    
    # --- Text features ---
    df["review_length"] = df["review"].str.len()
    df["word_count"] = df["review"].str.split().str.len()
    
    # --- Drop original columns ---
    df = df.drop(columns=["name", "size", "join_date", "review"])
    
    return df

df_ml = transform_for_ml(df)
print(f"ML-ready shape: {df_ml.shape}")
df_ml

## 10. Save Transformed Data

In [ ]:
import os

os.makedirs("data", exist_ok=True)
df_ml.to_csv("data/employees_transformed.csv", index=False)
print("Saved to data/employees_transformed.csv")

## Exercises

### Exercise 1: Compare scaling methods

Apply both MinMaxScaler and StandardScaler to the salary column. Compare the distributions:
- What are the min/max values for each?
- Which method preserves the original distribution shape better?
- When would you choose one over the other?

### Exercise 2: Encode a new dataset

Use this dataset and apply appropriate encoding:

```python
products = pd.DataFrame({
    "name": ["A", "B", "C", "D", "E"],
    "category": ["Electronics", "Clothing", "Electronics", "Food", "Clothing"],
    "rating": ["low", "medium", "high", "medium", "low"],
    "price": [99.99, 49.99, 199.99, 9.99, 29.99]
})
```

Tasks:
1. One-hot encode `category`
2. Label encode `rating` with correct ordinal order
3. Normalize `price`
4. Create a `price_per_rating` feature

### Exercise 3: Text processing chain

Process these reviews through the full pipeline:

```python
reviews = [
    "This product is AMAZING!! Best purchase ever.",
    "Terrible quality... waste of money.",
    "It's okay, nothing special but not bad either."
]
```

Steps:
1. Lowercase
2. Remove punctuation
3. Tokenize
4. Remove stopwords
5. Count words in each processed review

In [ ]:
# Your code here for Exercise 1
# ...